# Feature Engineering — Account Churn Prediction

Aggregates users/events/support per account and joins account attributes
to build a per-account feature table for churn classification.

**Reads:** `silver_accounts`, `silver_users`, `silver_events`, `silver_support_tickets`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, count, avg, countDistinct,
    sum as spark_sum
)

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
acc = spark.read.table('silver_accounts')
usr = spark.read.table('silver_users')
ev = spark.read.table('silver_events')
tk = spark.read.table('silver_support_tickets')
print(f'accounts={acc.count():,} users={usr.count():,} events={ev.count():,} tickets={tk.count():,}')

required = {'account_id', 'is_churned', 'health_score', 'mrr_usd'}
missing = required - set(acc.columns)
if missing:
    raise ValueError(f'silver_accounts missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Per-account aggregates
u = usr.groupBy('account_id').agg(
    count('*').alias('user_count'),
    spark_sum('is_active').alias('active_user_count'),
    avg('logins_last_30_days').alias('avg_logins_30d'))
e = ev.groupBy('account_id').agg(
    count('*').alias('event_count'),
    countDistinct('feature').alias('distinct_features'),
    avg('duration_secs').alias('avg_duration'))
t = tk.groupBy('account_id').agg(
    count('*').alias('ticket_count'),
    spark_sum('is_sla_breached').alias('sla_breach_count'),
    avg('csat_score').alias('avg_csat'),
    avg('resolution_hrs').alias('avg_resolution_hrs'))

In [ ]:
# Join account attributes + aggregates. EXCLUDE churn_date leakage; label = is_churned.
ml_features = (
    acc.select(
        'account_id', 'plan', 'industry', 'region',
        'mrr_usd', 'seat_count', 'health_score', 'tenure_days',
        col('is_churned').alias('had_churn'),
    )
    .join(u, 'account_id', 'left')
    .join(e, 'account_id', 'left')
    .join(t, 'account_id', 'left')
    .na.fill(0, ['user_count', 'active_user_count', 'avg_logins_30d', 'event_count',
                 'distinct_features', 'avg_duration', 'ticket_count', 'sla_breach_count',
                 'avg_csat', 'avg_resolution_hrs'])
    .na.fill('unknown', subset=['plan', 'industry', 'region'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_churn') == 1).count()
churn_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 500 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} churn rows '
        f'({churn_rate:.2f}%). Check is_churned typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | churn rate {churn_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')